# Amazon ML Challenge 2026 — Entity Resolution
## Production Pipeline · Google Colab T4 GPU
**Only requirement:** Dataset uploaded to Google Drive at  
`MyDrive/Amazon-ML-entity-resolution/dataset/{train,test}/`

Run **Runtime → Run all** and the pipeline does everything else.

### Cell 1 · Environment Setup

In [ ]:
# ── Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm>=4.3.0', 'scikit-learn>=1.4.0', 'rapidfuzz>=3.8.0',
    'joblib>=1.4.0', 'psutil', 'catboost', 'xgboost',
    'sentence-transformers', 'networkx', 'jellyfish'])

# ── Core imports ───────────────────────────────────────────────────────────
import os, gc, re, time, shutil, unicodedata, logging, json as _json
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Set, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import joblib
import psutil
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
import networkx as nx

try:
    import jellyfish; HAS_JELLYFISH = True
except ImportError:
    HAS_JELLYFISH = False

try:
    import cupy as cp
    _ = cp.array([1.0])
    _free, _total = cp.cuda.runtime.memGetInfo()
    GPU_NAME = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    USE_GPU = True
    print(f'GPU  : {GPU_NAME}')
    print(f'VRAM : {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')
except Exception as _e:
    cp = None; USE_GPU = False
    print(f'No GPU CuPy ({_e}) — using CPU path')

# ── Paths (Colab-only) ─────────────────────────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution')
DRIVE_DATASET = DRIVE_ROOT / 'dataset'
LOCAL_DATASET = Path('/content/dataset')
ROOT          = Path('/content')

TRAIN_DIR  = LOCAL_DATASET / 'train'
TEST_DIR   = LOCAL_DATASET / 'test'
OUTPUT_DIR = ROOT / 'output'
MODEL_DIR  = ROOT / 'models'
CKPT_DIR   = ROOT / 'checkpoints'
REPORT_DIR = ROOT / 'reports' / 'error_analysis'

# ── Validate dataset is on Drive ───────────────────────────────────────────
if not DRIVE_DATASET.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DRIVE_DATASET}\n'
        'Upload your dataset folder to MyDrive/Amazon-ML-entity-resolution/dataset/'
    )

# ── Copy dataset Drive -> local SSD (fast I/O) ────────────────────────────
if not LOCAL_DATASET.exists():
    print('Copying dataset Drive -> local SSD (1-2 min)...')
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET, dirs_exist_ok=True)
    print('Dataset ready on local SSD.')
else:
    print('Dataset already on local SSD.')

# ── Create output directories ──────────────────────────────────────────────
for _d in [OUTPUT_DIR, MODEL_DIR, REPORT_DIR,
           CKPT_DIR/'preprocess', CKPT_DIR/'blocking', CKPT_DIR/'features']:
    _d.mkdir(parents=True, exist_ok=True)

DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT = DRIVE_ROOT / 'output'
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# ── Memory helpers ─────────────────────────────────────────────────────────
def free_gpu():
    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    gc.collect()

def report_memory(label=''):
    ram = psutil.virtual_memory().available / 1e9
    msg = f'{label}  RAM free: {ram:.1f} GB'
    if USE_GPU:
        f, t = cp.cuda.runtime.memGetInfo()
        msg += f'  |  VRAM: {f/1e9:.1f}/{t/1e9:.1f} GB'
    print(msg)

def save_to_drive(local_path: Path, subdir='checkpoints'):
    dest = DRIVE_ROOT / subdir / local_path.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(local_path, dest)

def load_from_drive(filename: str, subdir='checkpoints') -> Optional[Path]:
    local = CKPT_DIR / filename
    if local.exists(): return local
    remote = DRIVE_ROOT / subdir / filename
    if remote.exists():
        shutil.copy(remote, local)
        return local
    return None

print(f'System RAM : {psutil.virtual_memory().total/1e9:.1f} GB')
report_memory('Setup complete.')


### Cell 2 · Configuration — change settings here only

In [ ]:
# ==============================================================
# ALL SETTINGS IN ONE PLACE
# ==============================================================

# ── Experiment scope ───────────────────────────────────────────
SAMPLE_S1     = 50_000   # int → dev run │ None → full 2.2M training run
QUARTER_SPLIT = None     # 'Q1', 'Q2', 'Q3', 'Q4' or None. If set, loads only that quarter's S1.\nFULL_S23      = True     # True → full 10M S2/S3 │ False → sampled S2/S3
K_CANDIDATES  = 50       # top-K candidates per S1 entity
RANDOM_STATE  = 42

# ── TF-IDF ─────────────────────────────────────────────────────
TFIDF_ANALYZER     = 'char_wb'
TFIDF_NGRAM_RANGE  = (2, 4)
TFIDF_MAX_FEATURES = 50_000
TFIDF_MIN_DF       = 2

# ── T4 memory safety ───────────────────────────────────────────
# S1 dense batch:  300 x 50K x 4 bytes = 60 MB  (safe)
# S23 stays CSR sparse on CPU (~4 GB for 10M rows, ~2% density)
S1_BATCH_SIZE  = 300
MIN_SIM_SCORE  = 0.05

# ── Dataset split ──────────────────────────────────────────────
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# holdout = remaining 0.15 — check ONCE then freeze

# ── Thresholds ─────────────────────────────────────────────────
T_MATCH   = 0.55   # primary match threshold (swept on validation)
T_HIGH    = 0.75   # high-confidence gate for triangulation
T_SINGLE  = 0.65   # singleton guardrail floor
THRESHOLD_GRID = [round(x * 0.05, 2) for x in range(4, 20)]  # 0.20..0.95

# ── Ensemble ───────────────────────────────────────────────────
ENSEMBLE_WEIGHTS = {'lgbm': 0.5, 'catboost': 0.3, 'xgb': 0.2}

# ── LightGBM (CPU — more stable on Colab T4 than GPU mode) ────
LGBM_PARAMS = dict(
    objective='binary', metric='auc',
    num_leaves=127, learning_rate=0.05, n_estimators=1500,
    min_child_samples=20, subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
)
EARLY_STOPPING_ROUNDS = 50

# ── Embeddings ─────────────────────────────────────────────────
USE_EMBEDDINGS   = True
EMBED_BATCH_SIZE = 512
EMBED_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f'SAMPLE_S1={SAMPLE_S1}  QUARTER_SPLIT={QUARTER_SPLIT}  FULL_S23={FULL_S23}  K={K_CANDIDATES}')\nprint(f'Thresholds: T_MATCH={T_MATCH}  T_HIGH={T_HIGH}  T_SINGLE={T_SINGLE}')
report_memory('Config loaded.')


### Cell 3 · Text Normalization & Preprocessing

In [ ]:
# ==============================================================
# Preprocessing: NFKC + legal suffix expansion + field extraction
# Covers: French (SAS/SARL/SASU/EURL/SCI/GIE), Indian (PVT LTD/LLP/M/S), US/Global
# ==============================================================

_NAME_ABBREVS_RAW = {
    # US / International
    r'\bcorp\b':'corporation', r'\bco\b':'company', r'\binc\b':'incorporated',
    r'\bltd\b':'limited', r'\bllc\b':'limited liability company',
    r'\bllp\b':'limited liability partnership',
    r'\bpvt ltd\b':'private limited', r'\bprivate limited\b':'private limited',
    r'\bpvt\b':'private', r'\bpte ltd\b':'private limited', r'\bpte\b':'private',
    r'\blp\b':'limited partnership', r'\bplc\b':'public limited company',
    # French legal suffixes (open-set test entities)
    r'\bsas\b':'societe par actions simplifiee',
    r'\bsasu\b':'societe par actions simplifiee unipersonnelle',
    r'\bsarl\b':'societe a responsabilite limitee',
    r'\beurl\b':'entreprise unipersonnelle a responsabilite limitee',
    r'\bselarl\b':'societe exercice liberal a responsabilite limitee',
    r'\bsci\b':'societe civile immobiliere',
    r'\bgie\b':'groupement interet economique',
    r'\bsa\b':'societe anonyme',
    r'\bsnc\b':'societe en nom collectif',
    # Indian
    r'\bm/s\b':'messrs', r'\bms\b':'messrs',
    r'\bprop\b':'proprietor', r'\bproprietorship\b':'proprietor',
    # Common
    r'\b&\b':'and', r'\bsvc\b':'services', r'\bsvcs\b':'services',
    r'\btech\b':'technology', r'\bintl\b':'international',
    r'\bmfg\b':'manufacturing', r'\bmgmt\b':'management',
    r'\bassoc\b':'associates', r'\bnatl\b':'national', r'\bgrp\b':'group',
    r'\bent\b':'enterprises', r'\bhldgs\b':'holdings',
    r'\binds\b':'industries', r'\bsol\b':'solutions', r'\bsolns\b':'solutions',
}
_ADDR_ABBREVS_RAW = {
    r'\brd\b':'road', r'\bst\b':'street', r'\bave\b':'avenue',
    r'\bblvd\b':'boulevard', r'\bdr\b':'drive', r'\bln\b':'lane',
    r'\bct\b':'court', r'\bpl\b':'place', r'\bhwy\b':'highway',
    r'\bpkwy\b':'parkway', r'\bste\b':'suite', r'\bapt\b':'apartment',
    r'\bflr\b':'floor', r'\bbldg\b':'building',
    # French
    r'\brue\b':'rue', r'\bav\b':'avenue', r'\bbd\b':'boulevard',
    r'\bimp\b':'impasse', r'\bal\b':'allee', r'\bche\b':'chemin',
    r'\bres\b':'residence', r'\bcedex\b':'',
    # Indian
    r'\bnr\b':'near', r'\bopp\b':'opposite', r'\bsoc\b':'society',
}

def _compile(raw): return [(re.compile(p, re.IGNORECASE), r) for p, r in raw.items()]
_NAME_ABBREVS = _compile(_NAME_ABBREVS_RAW)
_ADDR_ABBREVS = _compile(_ADDR_ABBREVS_RAW)
_PUNCT_RE  = re.compile(r'[^\w\s]')
_SPACE_RE  = re.compile(r'\s+')
_DIGIT_RE  = re.compile(r'\d+')
_POSTAL_RE = re.compile(r'\b(\d{5,6}|[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b')
_HOUSE_RE  = re.compile(r'^\d+[A-Za-z]?\b')

def _safe(v) -> str:
    if v is None: return ''
    s = str(v).strip()
    return '' if s.lower() in ('nan', 'none', '') else s

def _apply(text, abbrevs):
    for pat, rep in abbrevs:
        text = pat.sub(rep, text)
    return text

def normalize_name(text) -> str:
    t = unicodedata.normalize('NFKC', _safe(text)).lower()
    t = _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', t)).strip()
    return _SPACE_RE.sub(' ', _apply(t, _NAME_ABBREVS)).strip()

def normalize_address(text) -> str:
    t = unicodedata.normalize('NFKC', _safe(text)).lower()
    t = _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', t)).strip()
    return _SPACE_RE.sub(' ', _apply(t, _ADDR_ABBREVS)).strip()

def extract_postal(text: str) -> str:
    m = _POSTAL_RE.search(_safe(text).upper())
    return m.group(0).replace(' ', '') if m else ''

def extract_house(text: str) -> str:
    m = _HOUSE_RE.match(_safe(text).strip())
    return m.group(0).lower() if m else ''

def phonetic_key(text: str) -> str:
    if not text: return ''
    toks = text.split()[:4]
    if HAS_JELLYFISH:
        return ' '.join(jellyfish.metaphone(t) for t in toks if t)
    # Soundex fallback
    def _sdx(s):
        s = s.upper()
        if not s: return ''
        tbl = str.maketrans('BFPVCGJKQSXZDTLMNR','111122222222334556')
        c = s[0] + s[1:].translate(tbl).replace('0','')
        d = c[0] + ''.join(x for i,x in enumerate(c[1:],1) if x!=c[i-1])
        return (d+'000')[:4]
    return ' '.join(_sdx(t) for t in toks if t)

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ('business_name','business_address','country'):
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str)
            df.loc[df[col].str.lower().isin(['nan','none']), col] = ''
    df['name_norm']     = df['business_name'].apply(normalize_name)
    df['addr_norm']     = df['business_address'].apply(normalize_address)
    df['country_norm']  = df['country'].apply(lambda x: _safe(x).lower().strip())
    df['addr_nums']     = df['business_address'].apply(
        lambda x: ' '.join(_DIGIT_RE.findall(_safe(x))))
    df['postal_code']   = df['business_address'].apply(extract_postal)
    df['house_num']     = df['business_address'].apply(extract_house)
    df['phonetic_name'] = df['name_norm'].apply(phonetic_key)
    df['blocking_text'] = df['name_norm']
    return df

print('Preprocessing module loaded.')


### Cell 4 · Data Loader

In [ ]:
def load_data(sample_s1=None, full_s23=False, quarter_split=None, split='train'):\n    """
    Load and preprocess train or test data.
    Returns: (s1_df, s2_df, s3_df, gt_dict)  [gt_dict=None for test]
    """
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    prefix = 'train' if split == 'train' else 'test'

    def _read(fname, id_col):
        df = pd.read_csv(base / fname, sep='\t', dtype=str, low_memory=False)
        df = df.fillna('')
        df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
        # Standardise entity_id column name
        for candidate in (id_col, 'entity_id', 'source1_entity_id',
                          'source2_entity_id', 'source3_entity_id', df.columns[0]):
            if candidate in df.columns:
                df = df.rename(columns={candidate: 'entity_id'})
                break
        # Standardise field names
        for old, new in [('name','business_name'),('address','business_address'),
                          ('business name','business_name'),('business address','business_address')]:
            if old in df.columns and new not in df.columns:
                df = df.rename(columns={old: new})
        return df

    print(f'Loading {split} data...')
    s1 = _read(f'{prefix}_source1.tsv', 'source1_entity_id')
    s2 = _read(f'{prefix}_source2.tsv', 'source2_entity_id')
    s3 = _read(f'{prefix}_source3.tsv', 'source3_entity_id')
\n    if quarter_split and split == 'train':\n        split_dir = DRIVE_ROOT / 'splits'\n        q_file = split_dir / f'{quarter_split}_s1_ids.txt'\n        if q_file.exists():\n            print(f'  Filtering S1 to {quarter_split}...', end=' ')\n            q_ids = set(pd.read_csv(q_file, header=None, dtype=str)[0])\n            s1 = s1[s1['entity_id'].isin(q_ids)].reset_index(drop=True)\n            print(f'kept {len(s1):,} rows.')\n        else:\n            print(f'  Warning: {q_file} not found. Proceeding with full S1.')\n\n
    if sample_s1 is not None:
        s1 = s1.sample(min(sample_s1, len(s1)), random_state=RANDOM_STATE).reset_index(drop=True)
    if not full_s23 and sample_s1 is not None:
        factor = max(1, len(s1) // 50_000)
        s2 = s2.sample(min(len(s1)*10, len(s2)), random_state=RANDOM_STATE).reset_index(drop=True)
        s3 = s3.sample(min(len(s1)*10, len(s3)), random_state=RANDOM_STATE).reset_index(drop=True)

    print(f'  Preprocessing...', end=' ')
    s1 = preprocess_df(s1)
    s2 = preprocess_df(s2)
    s3 = preprocess_df(s3)
    print('done.')

    gt_dict = None
    if split == 'train':
        gt_raw = pd.read_csv(TRAIN_DIR / 'train_ground_truth.tsv', sep='\t', dtype=str)
        gt_raw.columns = gt_raw.columns.str.strip().str.lower().str.replace(' ','_')
        s1_col  = [c for c in gt_raw.columns if 'source1' in c][0]
        s23_col = [c for c in gt_raw.columns if 'source1' not in c and 'entity' in c.lower()][0]
        s1_ids_set = set(s1['entity_id'])
        gt_dict = defaultdict(list)
        for _, row in gt_raw.iterrows():
            sid = str(row[s1_col]).strip()
            cid = str(row[s23_col]).strip()
            if sid in s1_ids_set:
                gt_dict[sid].append(cid)

    print(f'  S1:{len(s1):,}  S2:{len(s2):,}  S3:{len(s3):,}'
          + (f'  GT pairs:{sum(len(v) for v in gt_dict.values()):,}' if gt_dict else ''))
    report_memory('After load.')
    return s1, s2, s3, gt_dict

# ── Load training data ────────────────────────────────────────
s1, s2, s3, gt_dict = load_data(sample_s1=SAMPLE_S1, full_s23=FULL_S23, quarter_split=QUARTER_SPLIT, split='train')\n# Build combined S23 lookup used throughout
s23 = pd.concat([s2, s3], ignore_index=True)


### Cell 5 · Multi-Blocker Candidate Generation (Routes A–H)

In [ ]:
# ==============================================================
# 8-route independent blocker union
# A: Name TF-IDF   B: Addr TF-IDF   C: Exact name
# D: Rare token    E: Numeric/PIN    F: Full-doc TF-IDF
# G: Phonetic DM   H: Postal+House (DBA)
# ==============================================================

def _build_tfidf(texts):
    vec = TfidfVectorizer(analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM_RANGE,
                          max_features=TFIDF_MAX_FEATURES, min_df=TFIDF_MIN_DF,
                          sublinear_tf=True, norm='l2', dtype=np.float32)
    return vec, vec.fit_transform(texts)

def _cosine_topk(q_mat, s23_mat, s23_ids, k):
    """Memory-safe sparse-dense cosine (S1 dense batch x S23 sparse on CPU)."""
    results = defaultdict(list)
    for start in range(0, q_mat.shape[0], S1_BATCH_SIZE):
        batch = q_mat[start:start+S1_BATCH_SIZE].toarray().astype(np.float32)
        scores = (s23_mat @ batch.T).toarray()  # (N_s23, batch)
        for bi in range(scores.shape[1]):
            row = scores[:, bi]
            top_idx = np.argpartition(row, -min(k, len(row)))[-k:]
            for idx in top_idx:
                if row[idx] >= MIN_SIM_SCORE:
                    results[start+bi].append((s23_ids[idx], float(row[idx])))
    return results

def _inv_idx(df, col):
    idx = defaultdict(list)
    for eid, key in zip(df['entity_id'], df[col]):
        if key: idx[key].append(eid)
    return idx

def _rare_idx(df, col, max_df=10):
    freq = defaultdict(int)
    for t in df[col]:
        for w in t.split():
            if len(w) >= 3: freq[w] += 1
    rare = {w for w, c in freq.items() if 2 <= c <= max_df}
    idx = defaultdict(set)
    for eid, t in zip(df['entity_id'], df[col]):
        for w in t.split():
            if w in rare: idx[w].add(eid)
    return idx

def _num_idx(df):
    idx = defaultdict(set)
    for eid, nums in zip(df['entity_id'], df['addr_nums']):
        for n in nums.split():
            if len(n) >= 3: idx[n].add(eid)
    return idx

def _phonetic_idx(df):
    """Blocker G: per-token Double Metaphone/Soundex inverted index."""
    idx = defaultdict(set)
    for eid, name in zip(df['entity_id'], df['name_norm']):
        for tok in name.split():
            if len(tok) >= 3:
                key = jellyfish.metaphone(tok) if HAS_JELLYFISH else phonetic_key(tok)
                if key: idx[key].add(eid)
    return idx

def _postal_house_idx(df):
    """Blocker H: country-keyed postal+house index (catches DBA/trade-name same-address pairs)."""
    idx = defaultdict(set)
    for eid, postal, house, country in zip(
            df['entity_id'], df['postal_code'], df['house_num'], df['country_norm']):
        cc = (country[:2] if country else 'XX').upper()
        if postal and house:
            idx[f'{cc}|{postal}|{house}'].add(eid)
        if postal and len(postal) >= 5:
            idx[f'{cc}|{postal}'].add(eid)
    return idx

def generate_candidates(s1_df, s2_df, s3_df, k=None, gt_dict=None):
    k = k or K_CANDIDATES
    t0 = time.time()
    s1_ids  = list(s1_df['entity_id'])
    s23_df  = pd.concat([s2_df, s3_df], ignore_index=True)
    s23_ids = list(s23_df['entity_id'])
    raw: Dict[str, List] = {sid: [] for sid in s1_ids}

    def _tfidf_block(s1_texts, s23_texts, score_scale, k_route, label):
        print(f'  Blocker {label}...', end=' ')
        if sum(bool(t) for t in s23_texts) < 100:
            print('skipped (no text)')
            return
        all_texts = s1_texts + s23_texts
        vec, _ = _build_tfidf(all_texts)
        q_mat  = vec.transform(s1_texts)
        d_mat  = vec.transform(s23_texts)
        hits   = _cosine_topk(q_mat, d_mat, s23_ids, k_route)
        for i, sid in enumerate(s1_ids):
            for cid, sc in hits.get(i, []):
                raw[sid].append((cid, sc * score_scale))
        del q_mat, d_mat, hits; free_gpu()
        print('done')

    _tfidf_block(list(s1_df['blocking_text']), list(s23_df['blocking_text']),
                 1.0, k, 'A (name TF-IDF)')
    _tfidf_block(list(s1_df['addr_norm']), list(s23_df['addr_norm']),
                 0.80, k//2, 'B (addr TF-IDF)')
    _tfidf_block(
        [f"{n} {a}" for n,a in zip(s1_df['name_norm'], s1_df['addr_norm'])],
        [f"{n} {a}" for n,a in zip(s23_df['name_norm'], s23_df['addr_norm'])],
        0.75, k//3, 'F (full-doc TF-IDF)')

    print('  Blocker C (exact name)...', end=' ')
    ex_idx = _inv_idx(s23_df, 'name_norm')
    for sid, name in zip(s1_df['entity_id'], s1_df['name_norm']):
        for cid in ex_idx.get(name, [])[:20]: raw[sid].append((cid, 1.5))
    print('done')

    print('  Blocker D (rare token)...', end=' ')
    rare_idx = _rare_idx(s23_df, 'name_norm')
    for sid, name in zip(s1_df['entity_id'], s1_df['name_norm']):
        hits_d = set()
        for w in name.split():
            if len(w) >= 3: hits_d.update(rare_idx.get(w, set()))
        for cid in list(hits_d)[:15]: raw[sid].append((cid, 0.70))
    print('done')

    print('  Blocker E (numeric/PIN)...', end=' ')
    num_idx = _num_idx(s23_df)
    for sid, nums in zip(s1_df['entity_id'], s1_df['addr_nums']):
        hits_e = set()
        for n in nums.split():
            if len(n) >= 3: hits_e.update(num_idx.get(n, set()))
        for cid in list(hits_e)[:10]: raw[sid].append((cid, 0.60))
    print('done')

    print('  Blocker G (phonetic)...', end=' ')
    ph_idx = _phonetic_idx(s23_df)
    for sid, name in zip(s1_df['entity_id'], s1_df['name_norm']):
        hits_g = set()
        for tok in name.split():
            if len(tok) >= 3:
                key = jellyfish.metaphone(tok) if HAS_JELLYFISH else phonetic_key(tok)
                hits_g.update(ph_idx.get(key, set()))
        for cid in list(hits_g)[:15]: raw[sid].append((cid, 0.65))
    print('done')

    print('  Blocker H (postal+house DBA)...', end=' ')
    ph2_idx = _postal_house_idx(s23_df)
    for sid, postal, house, country in zip(
            s1_df['entity_id'], s1_df['postal_code'],
            s1_df['house_num'], s1_df['country_norm']):
        cc = (country[:2] if country else 'XX').upper()
        for cid in list(ph2_idx.get(f'{cc}|{postal}|{house}', set()))[:20]:
            raw[sid].append((cid, 1.3))
        for cid in list(ph2_idx.get(f'{cc}|{postal}', set()))[:10]:
            raw[sid].append((cid, 0.55))
    print('done')

    # Union + dedup + sort -> top-K
    final: Dict[str, List[str]] = {}
    cand_ranks: Dict[Tuple[str,str], int] = {}
    for sid in s1_ids:
        score_map: Dict[str, float] = {}
        for cid, sc in raw[sid]:
            score_map[cid] = max(score_map.get(cid, 0.0), sc)
        top = sorted(score_map.items(), key=lambda x: -x[1])[:k]
        final[sid] = [cid for cid, _ in top]
        for rank, (cid, _) in enumerate(top):
            cand_ranks[(sid, cid)] = rank

    elapsed = time.time() - t0
    avg_c = sum(len(v) for v in final.values()) / max(len(s1_ids), 1)
    diag = {'elapsed_sec': elapsed, 'avg_candidates': avg_c}

    if gt_dict:
        total_gt = sum(len(v) for v in gt_dict.values())
        found    = sum(1 for sid, gids in gt_dict.items()
                       for g in gids if g in set(final.get(sid,[])))
        recall   = found / total_gt * 100 if total_gt else 0.0
        diag.update({'candidate_recall': recall, 'found_gt': found, 'total_gt': total_gt})
        print(f'\n  === Candidate Recall: {recall:.2f}%  ({found}/{total_gt}) ===')

    print(f'  Avg cands/S1: {avg_c:.1f}  |  Elapsed: {elapsed:.1f}s')
    report_memory('After blocking.')
    return final, cand_ranks, diag

# ── Run blocking ──────────────────────────────────────────────
print('=== BLOCKING ===')
cands, cand_ranks, block_diag = generate_candidates(s1, s2, s3, gt_dict=gt_dict)


### Cell 6 · Feature Engineering (46 features)

In [ ]:
# ==============================================================
# 46 pairwise features
# ==============================================================
from rapidfuzz.distance import Levenshtein as _Lev

FEATURE_NAMES = [
    # Name string (6)
    'name_ratio','name_token_sort','name_token_set','name_partial','name_jw','name_jaccard',
    # Address string (5)
    'addr_ratio','addr_token_sort','addr_token_set','addr_jaccard','addr_num_overlap',
    # Global (1)
    'country_match',
    # Blocking signal (1)
    'blocking_rank',
    # Length (4)
    'name_len_s1','name_len_s23','name_len_ratio','addr_len_ratio',
    # Full document (3)
    'doc_ratio','doc_token_set','doc_jw',
    # Structural (8)
    'name_x_addr','min_name_addr','max_name_addr',
    'both_addr','missing_asym','exact_name','conflict_digits','strong_name_num',
    # Hard negative (5)
    'name_num_overlap','name_conflict_digits','addr_jw','name_len_diff','addr_len_diff',
    # Ranking (2)
    'cand_rank','is_s3',
    # Phonetic (2)
    'phonetic_jw','phonetic_lev',
    # Granular address (3)
    'postal_exact','house_exact','street_token_jaccard',
    # Asymmetry (1)
    'addr_detail_asymmetry',
    # DBA / franchise (2)
    'same_postal_diff_name','name_addr_conflict',
    # Embedding (1)
    'embed_cosine',
]
assert len(FEATURE_NAMES) == 46, f'Expected 46, got {len(FEATURE_NAMES)}'

_EMBED_CACHE: Dict[str, np.ndarray] = {}

def _r(a,b):   return fuzz.ratio(a,b)/100.0
def _ts(a,b):  return fuzz.token_sort_ratio(a,b)/100.0
def _tse(a,b): return fuzz.token_set_ratio(a,b)/100.0
def _pr(a,b):  return fuzz.partial_ratio(a,b)/100.0
def _jw(a,b):  return JaroWinkler.similarity(a,b)
def _lev(a,b):
    ml = max(len(a),len(b))
    return 1.0 - _Lev.distance(a,b)/ml if ml>0 else 1.0
def _jac(a,b):
    sa,sb = set(a.split()),set(b.split())
    return len(sa&sb)/max(len(sa|sb),1)
def _nums(t): return set(_DIGIT_RE.findall(t))
def _nov(a,b):
    na,nb = _nums(a),_nums(b)
    if not na or not nb: return 0.0
    return len(na&nb)/max(len(na|nb),1)
def _lr(a,b):
    la,lb=len(a),len(b)
    return min(la,lb)/max(la,lb) if max(la,lb)>0 else 1.0
def _street_tok(addr): return {t for t in addr.split() if not t.isdigit() and len(t)>1}
def _embed_cos(eid1,eid2):
    e1,e2 = _EMBED_CACHE.get(eid1), _EMBED_CACHE.get(eid2)
    if e1 is None or e2 is None: return 0.0
    return float(max(0.0,min(1.0,np.dot(e1,e2))))

def compute_features(s1id, s23id,
                     n1,a1,c1,p1,h1,ph1,
                     n2,a2,c2,p2,h2,ph2,
                     rank) -> List[float]:
    # Name (6)
    f_nr,f_nts,f_ntse,f_np = _r(n1,n2),_ts(n1,n2),_tse(n1,n2),_pr(n1,n2)
    f_njw,f_nj = _jw(n1,n2),_jac(n1,n2)
    # Address (5)
    f_ar,f_ats,f_atse = _r(a1,a2),_ts(a1,a2),_tse(a1,a2)
    f_aj = _jac(a1,a2)
    f_ano = _nov(a1,a2)
    # Global (1)
    f_cm = 1.0 if (c1 and c1==c2) else 0.0
    # Blocking (1)
    f_bs = float(rank)
    # Length (4)
    f_nls1,f_nls23 = float(len(n1)),float(len(n2))
    f_nlr = _lr(n1,n2)
    f_alr = _lr(a1,a2)
    # Full doc (3)
    d1,d2 = f'{n1} {a1} {c1}'.strip(), f'{n2} {a2} {c2}'.strip()
    f_dr,f_dtse,f_djw = _r(d1,d2),_tse(d1,d2),_jw(d1,d2)
    # Structural (8)
    f_nxa = f_ntse*f_atse
    f_mina,f_maxa = min(f_ntse,f_atse),max(f_ntse,f_atse)
    l1,l2 = len(a1),len(a2)
    f_ba = 1.0 if (l1>=5 and l2>=5) else 0.0
    f_ma = 1.0 if ((l1>=10 and l2<3) or (l2>=10 and l1<3)) else 0.0
    f_en = 1.0 if (n1 and n1==n2) else 0.0
    d1s,d2s = _nums(a1),_nums(a2)
    f_cd = 1.0 if (d1s and d2s and not(d1s&d2s)) else 0.0
    f_snn= 1.0 if (f_ntse>=0.85 and f_ano>=0.5) else 0.0
    # Hard negative (5)
    f_nno = _nov(n1,n2)
    nd1,nd2 = _nums(n1),_nums(n2)
    f_ncd = 1.0 if (nd1 and nd2 and not(nd1&nd2)) else 0.0
    f_ajw = _jw(a1,a2)
    f_nld = float(abs(len(n1)-len(n2)))
    f_ald = float(abs(l1-l2))
    # Ranking (2)
    f_cr  = float(rank)
    f_is3 = 1.0 if s23id.startswith('S3-') else 0.0
    # Phonetic (2)
    f_phjw  = _jw(ph1,ph2)  if (ph1 and ph2) else 0.0
    f_phlev = _lev(ph1,ph2) if (ph1 and ph2) else 0.0
    # Granular address (3)
    f_pe = 1.0 if (p1 and p1==p2) else 0.0
    f_he = 1.0 if (h1 and h1==h2) else 0.0
    st1,st2 = _street_tok(a1),_street_tok(a2)
    f_stj = len(st1&st2)/max(len(st1|st2),1)
    # Asymmetry (1)
    f_asym = 1.0 if (len(a1.split())>=4) != (len(a2.split())>=4) else 0.0
    # DBA / franchise (2)
    f_spdn = 1.0 if (f_pe==1.0 and f_ntse<0.5) else 0.0
    f_nac  = 1.0 if (f_ntse>=0.85 and f_cd==1.0) else 0.0
    # Embedding (1) — filled post-hoc by embed cell
    f_ec   = _embed_cos(s1id, s23id)

    return [f_nr,f_nts,f_ntse,f_np,f_njw,f_nj,
            f_ar,f_ats,f_atse,f_aj,f_ano,
            f_cm,f_bs,
            f_nls1,f_nls23,f_nlr,f_alr,
            f_dr,f_dtse,f_djw,
            f_nxa,f_mina,f_maxa,f_ba,f_ma,f_en,f_cd,f_snn,
            f_nno,f_ncd,f_ajw,f_nld,f_ald,
            f_cr,f_is3,
            f_phjw,f_phlev,
            f_pe,f_he,f_stj,
            f_asym,
            f_spdn,f_nac,
            f_ec]

# 6-tuple lookup: (name_norm, addr_norm, country_norm, postal, house, phonetic_name)
def build_lookup(df: pd.DataFrame) -> Dict:
    return {
        r['entity_id']: (
            r['name_norm'], r['addr_norm'], r['country_norm'],
            r.get('postal_code',''), r.get('house_num',''), r.get('phonetic_name','')
        )
        for _, r in df.iterrows()
    }

def build_feature_matrix(pairs, s1_lu, s23_lu, cand_ranks):
    from joblib import Parallel, delayed
    def _row(s1id, s23id):
        t1 = s1_lu.get(s1id,  ('','','','','',''))
        t2 = s23_lu.get(s23id, ('','','','','',''))
        rank = cand_ranks.get((s1id,s23id), K_CANDIDATES)
        return compute_features(s1id,s23id,*t1,*t2,rank)
    rows = Parallel(n_jobs=-1, prefer='threads')(
        delayed(_row)(s,c) for s,c in pairs)
    return np.array(rows, dtype=np.float32)

print(f'Feature module loaded. Features: {len(FEATURE_NAMES)}')


### Cell 7 · Evaluation Utilities

In [ ]:
def f05(precision, recall):
    d = 0.25*precision + recall
    return 1.25*precision*recall/d if d>0 else 0.0

def entity_f05(pred_set, gt_set):
    if not gt_set:   # true singleton
        return 1.0 if not pred_set else 0.0
    if not pred_set: # missed all
        return 0.0
    tp  = len(pred_set & gt_set)
    prec = tp/len(pred_set)
    rec  = tp/len(gt_set)
    return f05(prec, rec)

def evaluate(pred_dict, gt_dict, s1_ids):
    scores = []
    for sid in s1_ids:
        ps = set(pred_dict.get(sid,[]))
        gs = set(gt_dict.get(sid,[]))
        scores.append(entity_f05(ps,gs))
    macro_f05 = float(np.mean(scores))
    # Precision / recall at macro level
    plist, rlist = [], []
    for sid in s1_ids:
        ps = set(pred_dict.get(sid,[]))
        gs = set(gt_dict.get(sid,[]))
        if not gs: continue
        tp = len(ps&gs)
        plist.append(tp/len(ps) if ps else 0.0)
        rlist.append(tp/len(gs))
    return {'macro_f05': macro_f05,
            'macro_precision': float(np.mean(plist)) if plist else 0.0,
            'macro_recall': float(np.mean(rlist)) if rlist else 0.0,
            'n': len(scores)}

def threshold_sweep(pairs, labels, probs, gt_dict, s1_ids_set,
                    cands, grid=None):
    grid = grid or THRESHOLD_GRID
    best_thr, best_f05 = 0.5, 0.0
    best_pred = {}
    for thr in grid:
        pred = defaultdict(list)
        for (sid,cid), p in zip(pairs, probs):
            if p >= thr: pred[sid].append(cid)
        for sid in s1_ids_set:
            if sid not in pred: pred[sid] = []
        stats = evaluate(pred, gt_dict, list(s1_ids_set))
        if stats['macro_f05'] > best_f05:
            best_f05, best_thr, best_pred = stats['macro_f05'], thr, dict(pred)
    return best_thr, best_f05, best_pred

print('Evaluation utilities loaded.')


### Cell 8 · Multilingual Embeddings (MiniLM — optional, ~5 min on T4)

In [ ]:
if USE_EMBEDDINGS:
    import torch
    from sentence_transformers import SentenceTransformer
    _ck_key = f'embed_{SAMPLE_S1}_{FULL_S23}'
    _ck_emb = CKPT_DIR/'features'/f'{_ck_key}.npy'
    _ck_ids = CKPT_DIR/'features'/f'{_ck_key}_ids.pkl'
    all_df  = pd.concat([s1, s23], ignore_index=True)
    if _ck_emb.exists() and _ck_ids.exists():
        print('Loading embeddings from checkpoint...')
        _embs = np.load(_ck_emb)
        _ids  = joblib.load(_ck_ids)
    else:
        print(f'Encoding {len(all_df):,} entities with MiniLM...')
        _model = SentenceTransformer(EMBED_MODEL_NAME)
        _texts = [f"{n} {a}".strip() for n,a in zip(all_df['name_norm'], all_df['addr_norm'])]
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        _embs = _model.encode(_texts, batch_size=EMBED_BATCH_SIZE,
                               normalize_embeddings=True, device=device,
                               show_progress_bar=True, convert_to_numpy=True)
        _ids  = list(all_df['entity_id'])
        np.save(_ck_emb, _embs)
        joblib.dump(_ids, _ck_ids)
        save_to_drive(_ck_emb, 'checkpoints/features')
        del _model; free_gpu()
    _EMBED_CACHE.clear()
    for eid, emb in zip(_ids, _embs):
        _EMBED_CACHE[eid] = emb.astype(np.float32)
    del _embs
    print(f'Embedding cache: {len(_EMBED_CACHE):,} entities.')
    report_memory('After embeddings.')
else:
    print('Embeddings disabled (USE_EMBEDDINGS=False). Feature embed_cosine = 0.')


### Cell 9 · Ensemble Training (LightGBM + CatBoost + XGBoost)

In [ ]:
from catboost import CatBoostClassifier
import xgboost as xgb

# ── Split S1 entities 70/15/15 by entity_id ──────────────────
all_s1_ids = list(s1['entity_id'])
np.random.seed(RANDOM_STATE)
shuffled   = np.random.permutation(all_s1_ids)
n_tr  = int(len(shuffled)*TRAIN_FRAC)
n_va  = int(len(shuffled)*VAL_FRAC)
ids_tr, ids_va, ids_ho = (
    list(shuffled[:n_tr]),
    list(shuffled[n_tr:n_tr+n_va]),
    list(shuffled[n_tr+n_va:])
)
s1_tr = s1[s1['entity_id'].isin(set(ids_tr))].reset_index(drop=True)
s1_va = s1[s1['entity_id'].isin(set(ids_va))].reset_index(drop=True)
s1_ho = s1[s1['entity_id'].isin(set(ids_ho))].reset_index(drop=True)
print(f'Split: train={len(s1_tr):,}  val={len(s1_va):,}  holdout={len(s1_ho):,}')

# ── Build lookup tables ───────────────────────────────────────
s1_lu  = build_lookup(s1)
s23_lu = build_lookup(s23)

# ── Hard-negative mining: build pairs for training ────────────
def _mine_pairs(s1_sub, split_gt):
    pos, hard_neg, easy_neg = [], [], []
    for sid in s1_sub['entity_id']:
        gt_set = set(split_gt.get(sid, []))
        n1 = s1_lu.get(sid, ('','','','','',''))[0]
        c1 = s1_lu.get(sid, ('','','','','',''))[2]
        for cid in cands.get(sid, []):
            n2 = s23_lu.get(cid, ('','','','','',''))[0]
            c2 = s23_lu.get(cid, ('','','','','',''))[2]
            lbl = 1 if cid in gt_set else 0
            if lbl == 1:
                pos.append((sid, cid, 1))
            elif fuzz.token_set_ratio(n1,n2)/100.0 >= 0.85 and c1==c2:
                hard_neg.append((sid, cid, 0))
            else:
                easy_neg.append((sid, cid, 0))
    rng = np.random.default_rng(RANDOM_STATE)
    n_pos   = len(pos)
    n_hard  = min(n_pos*5, len(hard_neg))
    n_easy  = max(0, n_pos*5 - n_hard)
    if n_hard < len(hard_neg): hard_neg = [hard_neg[i] for i in rng.choice(len(hard_neg),n_hard,replace=False)]
    if n_easy < len(easy_neg):  easy_neg = [easy_neg[i]  for i in rng.choice(len(easy_neg), n_easy,replace=False)]
    all_p = pos+hard_neg+easy_neg
    rng.shuffle(all_p)
    pairs  = [(s,c) for s,c,_ in all_p]
    labels = [l for _,_,l in all_p]
    print(f'  pos={n_pos:,}  hard_neg={len(hard_neg):,}  easy_neg={len(easy_neg):,}  total={len(pairs):,}')
    return pairs, labels

print('Building training pairs...')
pairs_tr, y_tr = _mine_pairs(s1_tr, gt_dict)
print('Building validation pairs...')
pairs_va, y_va = _mine_pairs(s1_va, gt_dict)

print('Computing training features...')
X_tr = build_feature_matrix(pairs_tr, s1_lu, s23_lu, cand_ranks)
print('Computing validation features...')
X_va = build_feature_matrix(pairs_va, s1_lu, s23_lu, cand_ranks)
print(f'X_tr: {X_tr.shape}  X_va: {X_va.shape}')
report_memory('After feature matrices.')

# ── Train ensemble ────────────────────────────────────────────
print('\n[1/3] LightGBM...')
_p = {k:v for k,v in LGBM_PARAMS.items() if k!='n_estimators'}
dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_NAMES)
dval   = lgb.Dataset(X_va, label=y_va, feature_name=FEATURE_NAMES, reference=dtrain)
lgbm_model = lgb.train(
    _p, dtrain, num_boost_round=LGBM_PARAMS['n_estimators'],
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS,verbose=False),
               lgb.log_evaluation(200)])
print(f'  Best iteration: {lgbm_model.best_iteration}')

print('[2/3] CatBoost...')
cb_model = CatBoostClassifier(
    iterations=1000, learning_rate=0.05, depth=8,
    l2_leaf_reg=3, eval_metric='AUC', use_best_model=True,
    early_stopping_rounds=50, verbose=200,
    task_type='CPU', random_seed=RANDOM_STATE)
cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), feature_names=FEATURE_NAMES)
print(f'  Best iteration: {cb_model.best_iteration_}')

print('[3/3] XGBoost...')
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='auc',
    max_depth=8, learning_rate=0.05, n_estimators=1000,
    subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, random_state=RANDOM_STATE,
    tree_method='hist', early_stopping_rounds=50, verbosity=0)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=200)
print(f'  Best iteration: {xgb_model.best_iteration}')

models = {'lgbm': lgbm_model, 'catboost': cb_model, 'xgb': xgb_model}

def ensemble_predict(models, X):
    """Geometric mean blend: P_lgbm^0.5 * P_cb^0.3 * P_xgb^0.2"""
    w = ENSEMBLE_WEIGHTS
    p1 = np.clip(models['lgbm'].predict(X), 1e-7, 1-1e-7)
    p2 = np.clip(models['catboost'].predict_proba(X)[:,1], 1e-7, 1-1e-7)
    p3 = np.clip(models['xgb'].predict_proba(X)[:,1], 1e-7, 1-1e-7)
    return p1**w['lgbm'] * p2**w['catboost'] * p3**w['xgb']

# ── Threshold sweep on validation ────────────────────────────
print('\nSweeping thresholds on validation set...')
va_probs = ensemble_predict(models, X_va)
best_thr, best_val_f05, _ = threshold_sweep(
    pairs_va, y_va, va_probs, gt_dict, set(ids_va), cands)
print(f'  Best threshold: {best_thr}  Val F0.5: {best_val_f05:.4f}')

# ── Holdout evaluation ────────────────────────────────────────
print('Building holdout pairs...')
pairs_ho, y_ho = _mine_pairs(s1_ho, gt_dict)
X_ho = build_feature_matrix(pairs_ho, s1_lu, s23_lu, cand_ranks)
ho_probs = ensemble_predict(models, X_ho)

pred_ho   = defaultdict(list)
prob_map_ho = {}
feat_map_ho = {}
for (sid,cid), p, feat_row in zip(pairs_ho, ho_probs, X_ho):
    prob_map_ho[(sid,cid)] = float(p)
    feat_map_ho[(sid,cid)] = dict(zip(FEATURE_NAMES, feat_row.tolist()))
    if p >= best_thr: pred_ho[sid].append(cid)
for sid in ids_ho:
    if sid not in pred_ho: pred_ho[sid] = []

ho_stats = evaluate(pred_ho, gt_dict, ids_ho)
print(f'  Holdout F0.5={ho_stats["macro_f05"]:.4f}  '
      f'P={ho_stats["macro_precision"]:.4f}  R={ho_stats["macro_recall"]:.4f}')

# ── Save models ───────────────────────────────────────────────
joblib.dump(models, MODEL_DIR/'ensemble_models.joblib')
meta = {'best_threshold': best_thr, 'val_f05': best_val_f05,
        'holdout_f05': ho_stats['macro_f05'],
        'n_features': len(FEATURE_NAMES), 'feature_names': FEATURE_NAMES,
        'sample_s1': SAMPLE_S1, 'k': K_CANDIDATES,
        'candidate_recall': block_diag.get('candidate_recall')}
joblib.dump(meta, MODEL_DIR/'meta.joblib')
save_to_drive(MODEL_DIR/'ensemble_models.joblib', 'models')
save_to_drive(MODEL_DIR/'meta.joblib', 'models')

# ── Export JSON summary ───────────────────────────────────────
with open(OUTPUT_DIR/'experiment_summary.json','w') as _f:
    _json.dump(meta, _f, indent=2, default=str)
print(_json.dumps(meta, indent=2, default=str))
report_memory('Training complete.')


### Cell 10 · Post-Processing (Graph Triangulation + Singleton Shield)

In [ ]:
# ==============================================================
# Post-Processing
# 1. NetworkX graph transitive closure (S2<->S3 triangulation)
# 2. Feature-aware singleton precision shield
# ==============================================================
_S2S3_IDX: Optional[Dict] = None

def _build_s2s3_idx(s2_df, s3_df, threshold=0.85):
    all_t = list(s2_df['blocking_text']) + list(s3_df['blocking_text'])
    vec, _ = _build_tfidf(all_t)
    s2_mat = vec.transform(list(s2_df['blocking_text']))
    s3_mat = vec.transform(list(s3_df['blocking_text']))
    s2_ids, s3_ids = list(s2_df['entity_id']), list(s3_df['entity_id'])
    out = defaultdict(list)
    for start in range(0, len(s2_ids), S1_BATCH_SIZE):
        batch  = s2_mat[start:start+S1_BATCH_SIZE].toarray().astype(np.float32)
        scores = (s3_mat @ batch.T).toarray()
        for bi in range(scores.shape[1]):
            row = scores[:, bi]
            for idx in np.where(row >= threshold)[0]:
                out[s2_ids[start+bi]].append(s3_ids[idx])
    free_gpu()
    return dict(out)

def graph_triangulate(s1_ids, pred_dict, prob_map, s23_lu,
                      s2_df, s3_df, t_high=T_HIGH, t_support=0.40):
    global _S2S3_IDX
    if _S2S3_IDX is None:
        print('  Building S2<->S3 index...', end=' ')
        _S2S3_IDX = _build_s2s3_idx(s2_df, s3_df)
        print(f'{len(_S2S3_IDX):,} S2 nodes indexed')
    G = nx.Graph()
    n_boost = 0
    for s1id in s1_ids:
        G.add_node(s1id)
        cand_list = pred_dict.get(s1id, [])
        for cid in cand_list:
            p = prob_map.get((s1id,cid), 0.0)
            if p >= t_high:
                G.add_edge(s1id, cid, weight=p)
        s2c = [c for c in cand_list if c.startswith('S2-')]
        s3c = [c for c in cand_list if c.startswith('S3-')]
        for s2id in s2c:
            p2 = prob_map.get((s1id,s2id), 0.0)
            for s3id in (_S2S3_IDX.get(s2id,[]) + s3c):
                p3 = prob_map.get((s1id,s3id), 0.0)
                if max(p2,p3) < t_support: continue
                t2 = s23_lu.get(s2id,('','','','','',''))
                t3 = s23_lu.get(s3id,('','','','','',''))
                nsim = fuzz.token_set_ratio(t2[0],t3[0])/100.0
                asim = fuzz.token_set_ratio(t2[1],t3[1])/100.0 if (t2[1] and t3[1]) else 1.0
                if nsim >= 0.88 and asim >= 0.70:
                    G.add_edge(s2id, s3id, weight=nsim*asim)
                    if p2 >= t_support: G.add_edge(s1id, s2id, weight=p2); n_boost+=1
                    if p3 >= t_support: G.add_edge(s1id, s3id, weight=p3); n_boost+=1
            # Pull S3 twins of high-confidence S2
            if p2 >= t_high:
                for s3t in _S2S3_IDX.get(s2id,[]):
                    if not G.has_edge(s1id,s3t):
                        G.add_edge(s1id, s3t, weight=0.0)
    refined = {sid:[] for sid in s1_ids}
    for comp in nx.connected_components(G):
        s1n  = [n for n in comp if n.startswith('S1-')]
        s23n = [n for n in comp if not n.startswith('S1-')]
        for sid in s1n: refined[sid].extend(s23n)
    print(f'  Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges  '
          f'|  boosted: {n_boost:,}')
    return refined

def singleton_shield(pred_dict, prob_map, feat_map, t_single=T_SINGLE):
    pruned = 0
    out = {}
    for sid, preds in pred_dict.items():
        if len(preds) == 1:
            cid  = preds[0]
            p    = prob_map.get((sid,cid), 0.0)
            feat = feat_map.get((sid,cid), {})
            if (p < t_single
                    or feat.get('conflict_digits',0.0) == 1.0
                    or feat.get('name_ratio',1.0) < 0.70):
                out[sid] = []; pruned += 1; continue
        out[sid] = preds
    print(f'  Singleton shield pruned: {pruned:,} lone marginal predictions')
    return out

def postprocess(s1_ids, pred_dict, prob_map, feat_map, s23_lu, s2_df, s3_df,
                triangulate=True, shield=True):
    result = dict(pred_dict)
    if triangulate:
        print('  Running graph triangulation...')
        result = graph_triangulate(s1_ids, result, prob_map, s23_lu, s2_df, s3_df)
    if shield:
        result = singleton_shield(result, prob_map, feat_map)
    return result

print('Post-processing module loaded.')


### Cell 11 · Missed-Match Analysis (optional diagnostic)

In [ ]:
print('=== Missed-Match Analysis ===')
missed_rows = []
s23_id_set = set(s23['entity_id'])
for sid, gt_ids in gt_dict.items():
    found = set(cands.get(sid, []))
    for gid in gt_ids:
        if gid not in found:
            t1 = s1_lu.get(sid, ('','','','','',''))
            t2 = s23_lu.get(gid, ('','','','','',''))
            nr = fuzz.token_set_ratio(t1[0],t2[0])/100.0
            in_s23 = gid in s23_id_set
            if nr >= 0.80: reason='NAME_SIMILAR_MISSED'
            elif not t1[1] or not t2[1]: reason='MISSING_ADDR'
            elif t1[2] != t2[2]: reason='COUNTRY_DIFF'
            elif nr < 0.30: reason='DBA_TRADE_NAME'
            else: reason='OTHER'
            missed_rows.append({'s1_id':sid,'gt_id':gid,
                                 's1_name':t1[0],'s23_name':t2[0],
                                 's1_addr':t1[1],'s23_addr':t2[1],
                                 'name_sim':round(nr,3),'reason':reason,
                                 'in_s23':in_s23})
missed_df = pd.DataFrame(missed_rows)
if len(missed_df):
    print(missed_df['reason'].value_counts().to_string())
    out_path = REPORT_DIR / 'missed_blocking_matches.csv'
    missed_df.to_csv(out_path, index=False)
    save_to_drive(out_path, 'reports')
    print(f'\nSaved {len(missed_df):,} missed pairs -> {out_path}')
else:
    print('No missed matches (candidate recall = 100%!)')


### Cell 12 · Full Test Inference

In [ ]:
print('=== FULL TEST INFERENCE ===')

# ── Load test data ────────────────────────────────────────────
s1_test, s2_test, s3_test, _ = load_data(
    sample_s1=None, full_s23=True, split='test')
s23_test    = pd.concat([s2_test, s3_test], ignore_index=True)
s1_lu_test  = build_lookup(s1_test)
s23_lu_test = build_lookup(s23_test)

# ── Add test entities to embedding cache if needed ────────────
if USE_EMBEDDINGS and _EMBED_CACHE:
    _missing = [r for _, r in s23_test.iterrows() if r['entity_id'] not in _EMBED_CACHE]
    if _missing:
        import torch
        from sentence_transformers import SentenceTransformer
        print(f'Encoding {len(_missing):,} new test entities...')
        _m = SentenceTransformer(EMBED_MODEL_NAME)
        _t = [f"{r['name_norm']} {r['addr_norm']}".strip() for r in _missing]
        _e = _m.encode(_t, batch_size=EMBED_BATCH_SIZE,
                        normalize_embeddings=True,
                        device='cuda' if torch.cuda.is_available() else 'cpu',
                        convert_to_numpy=True)
        for r, emb in zip(_missing, _e):
            _EMBED_CACHE[r['entity_id']] = emb.astype(np.float32)
        del _m, _e; free_gpu()

# ── Blocking ──────────────────────────────────────────────────
print('\nBlocking test S1 against full S2/S3...')
cands_test, crand_test, _ = generate_candidates(
    s1_test, s2_test, s3_test, k=K_CANDIDATES)

# ── Load trained ensemble ─────────────────────────────────────
models_inf = joblib.load(MODEL_DIR/'ensemble_models.joblib')
meta_inf   = joblib.load(MODEL_DIR/'meta.joblib')
thr        = meta_inf['best_threshold']
print(f'Loaded ensemble. Threshold: {thr}')

# ── Batch inference ───────────────────────────────────────────
INFER_BATCH = 500_000   # pairs per batch
all_s1_test = list(s1_test['entity_id'])
pred_test: Dict[str, List[str]] = {sid: [] for sid in all_s1_test}
prob_map_test: Dict[Tuple[str,str], float] = {}
feat_map_test: Dict[Tuple[str,str], Dict]  = {}

all_pairs_test = [(sid, cid)
                  for sid in all_s1_test
                  for cid in cands_test.get(sid, [])]

print(f'Total inference pairs: {len(all_pairs_test):,}')
for start in range(0, len(all_pairs_test), INFER_BATCH):
    batch_pairs = all_pairs_test[start:start+INFER_BATCH]
    X_batch = build_feature_matrix(batch_pairs, s1_lu_test, s23_lu_test, crand_test)
    probs   = ensemble_predict(models_inf, X_batch)
    for (sid,cid), p, feat_row in zip(batch_pairs, probs, X_batch):
        prob_map_test[(sid,cid)] = float(p)
        feat_map_test[(sid,cid)] = dict(zip(FEATURE_NAMES, feat_row.tolist()))
        if p >= thr: pred_test[sid].append(cid)
    pct = min(start+INFER_BATCH, len(all_pairs_test))/len(all_pairs_test)*100
    print(f'  {pct:.0f}%', end='\r')

print(f'\nPre-postprocess: {sum(len(v)>0 for v in pred_test.values()):,} S1 with >=1 match')

# ── Post-processing ───────────────────────────────────────────
print('Applying post-processing...')
_S2S3_IDX = None   # reset so it's rebuilt on test S2/S3
pred_test  = postprocess(
    all_s1_test, pred_test, prob_map_test, feat_map_test,
    s23_lu_test, s2_test, s3_test)

print(f'Post-postprocess: {sum(len(v)>0 for v in pred_test.values()):,} S1 with >=1 match')

# ── Generate candidate_pairs.tsv ──────────────────────────────
cp_path = OUTPUT_DIR / 'candidate_pairs.tsv'
with open(cp_path, 'w') as _f:
    _f.write('source1_entity_id\tcandidate_entity_id\n')
    for sid in all_s1_test:
        for cid in cands_test.get(sid, []):
            _f.write(f'{sid}\t{cid}\n')
print(f'candidate_pairs.tsv written: {cp_path}')

# ── Generate matching_results.tsv ─────────────────────────────
mr_path = OUTPUT_DIR / 'matching_results.tsv'
with open(mr_path, 'w') as _f:
    _f.write('source1_entity_id\tmatched_entity_ids\n')
    for sid in all_s1_test:
        matched = ','.join(pred_test.get(sid, []))
        _f.write(f'{sid}\t{matched}\n')
print(f'matching_results.tsv written: {mr_path}')

# ── Save outputs to Drive ─────────────────────────────────────
shutil.copy(cp_path, DRIVE_OUTPUT/'candidate_pairs.tsv')
shutil.copy(mr_path, DRIVE_OUTPUT/'matching_results.tsv')
print('Both files saved to Drive.')
report_memory('Inference complete.')


### Cell 13 · Submission Validator (embedded — no external files needed)

In [ ]:
# ==============================================================
# Inline submission validator — no external utils/validate_submission.py
# Checks schema, coverage, and candidate superset requirement
# ==============================================================
print('=== SUBMISSION VALIDATION ===')
errors = []

# 1. Read outputs
try:
    df_mr = pd.read_csv(OUTPUT_DIR/'matching_results.tsv', sep='\t', dtype=str).fillna('')
    df_cp = pd.read_csv(OUTPUT_DIR/'candidate_pairs.tsv',  sep='\t', dtype=str).fillna('')
except FileNotFoundError as e:
    raise RuntimeError(f'Output file missing: {e}. Run Cell 12 first.')

# 2. Schema check
assert 'source1_entity_id' in df_mr.columns, 'matching_results.tsv missing source1_entity_id column'
assert 'matched_entity_ids' in df_mr.columns, 'matching_results.tsv missing matched_entity_ids column'
assert 'source1_entity_id' in df_cp.columns, 'candidate_pairs.tsv missing source1_entity_id column'
assert 'candidate_entity_id' in df_cp.columns, 'candidate_pairs.tsv missing candidate_entity_id column'
print('  [PASS] Schema correct')

# 3. All test S1 entities present in matching_results
test_s1_ids = set(s1_test['entity_id'])
mr_s1_ids   = set(df_mr['source1_entity_id'])
missing_s1  = test_s1_ids - mr_s1_ids
if missing_s1:
    errors.append(f'  [FAIL] {len(missing_s1):,} test S1 entities missing from matching_results.tsv')
else:
    print(f'  [PASS] All {len(test_s1_ids):,} test S1 entities present in matching_results.tsv')

# 4. No duplicate S1 rows
dups = df_mr['source1_entity_id'].duplicated().sum()
if dups:
    errors.append(f'  [FAIL] {dups:,} duplicate S1 rows in matching_results.tsv')
else:
    print('  [PASS] No duplicate S1 rows')

# 5. Matched IDs must be subset of candidate_pairs
cp_pairs = set(zip(df_cp['source1_entity_id'], df_cp['candidate_entity_id']))
not_in_cp = 0
for _, row in df_mr.iterrows():
    sid = row['source1_entity_id']
    mids = [m for m in str(row['matched_entity_ids']).split(',') if m.strip()]
    for mid in mids:
        if (sid, mid) not in cp_pairs:
            not_in_cp += 1
if not_in_cp:
    errors.append(f'  [FAIL] {not_in_cp:,} matched IDs not in candidate_pairs.tsv')
else:
    print('  [PASS] All matched IDs are in candidate_pairs.tsv')

# 6. Statistics
n_with_match = (df_mr['matched_entity_ids'].str.strip() != '').sum()
n_singleton  = (df_mr['matched_entity_ids'].str.strip() == '').sum()
print(f'  Stats: {n_with_match:,} entities with match  |  {n_singleton:,} predicted singletons')
print(f'  candidate_pairs.tsv: {len(df_cp):,} rows')

# ── Final verdict ─────────────────────────────────────────────
if errors:
    print('\n=== VALIDATION FAILED ===')
    for e in errors: print(e)
else:
    print('\n=== VALIDATION PASSED ===')
    print('Both submission files are ready in /content/output/ and Drive/output/')
    print(f'  {OUTPUT_DIR}/candidate_pairs.tsv')
    print(f'  {OUTPUT_DIR}/matching_results.tsv')
